# UABR — Phases 6 + 7: SFT with replay, then the one test evaluation

**Kaggle settings:** Accelerator **GPU T4 x2** · Internet **On** · **+ Add Input → Your Work**:
- the Phase 2 v2 notebook (`data_v2`)
- the Phase 4 notebook (`delegation_v1.pt` + `facts_knowledge.parquet`)
- the Phase 5 notebook (`runs/sft_standard_s11/final.pt`, `runs/sft_standard_s29/final.pt` = the **Standard SFT** branches)

**What happens:**
1. **Replay table:** `delegation_v1` scores every `replay_construction` prompt once (call score, reward of answering, reward of calling).
2. **Selection:** Random, BPR and UABR each pick replay examples worth the same number of tokens (`configs/replay.yaml`).
3. **SFT + replay:** 3 methods × 2 seeds = 6 runs. Same instruction batches as Standard SFT for that seed; only the added replay examples differ.
4. **Test evaluation (opened once):** `delegation_v1` + 8 branches on all test prompts; paired bootstrap 95% CIs.

`QUICK = True` → Run All (~20 min) checks the pipeline. Then `QUICK = False` → **Save Version › Save & Run All (Commit)** (~2–3 h).

In [ ]:
QUICK = True
DATA_VERSION = "data_v2"
REPO_URL = "https://github.com/GTX-Pradeep/uabr-slm.git"

import os, glob, json, subprocess
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/uabr_slm"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

def find(pattern, what):
    hits = [p for p in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True) if "_quick" not in p]
    assert hits, f"{pattern} not found: Add Input -> Your Work -> {what}"
    return hits

DATA = os.path.dirname(find(f"{DATA_VERSION}/manifest.json", "the Phase 2 v2 notebook")[0])
DELEG = [p for p in find("delegation_v1.pt", "the Phase 4 notebook") if "/runs/" not in p][0]
KNOW = find("facts_knowledge.parquet", "the Phase 4 notebook")[0]
SEEDS, METHODS = [11, 29], ["random", "bpr", "uabr"]
STD = {s: find(f"sft_standard_s{s}/final.pt", "the Phase 5 notebook")[0] for s in SEEDS}
INSTR_TOKENS = json.load(open(os.path.join(os.path.dirname(STD[SEEDS[0]]), "budget.json")))["instruction"]["tokens"]
SUF = "_quick" if QUICK else ""
REPLAY, FINAL = f"{WORK}/runs/replay{SUF}", f"{WORK}/runs/final{SUF}"
print("DATA :", DATA); print("DELEG:", DELEG); print("KNOW :", KNOW); print("STD  :", STD)
print("instruction tokens per SFT run:", f"{INSTR_TOKENS:,}")
!git log --oneline -1
!python -m pytest tests -q -x

## 1. Replay table (~3 min)
One row per `replay_construction` prompt (people never used in Phase 4 training, dev or test).

In [ ]:
!python scripts/build_replay_table.py --delegation {DELEG} --data {DATA} --knowledge {KNOW} --out {REPLAY}/replay_table.parquet

## 2. Select replay examples (seconds)
Every method gets the same replay-token budget (5% of the instruction tokens). The report shows how different the three selections are.

In [ ]:
!python scripts/select_replay.py --table {REPLAY}/replay_table.parquet --instruction_tokens {INSTR_TOKENS} --out {REPLAY}

## 3. SFT + replay: 6 runs (~15 min each in full mode)
If the session stops early, re-run this cell: `--resume` continues from the last checkpoint.

In [ ]:
quick = "--quick" if QUICK else ""
for seed in SEEDS:
    for m in METHODS:
        run = f"{WORK}/runs/sft_{m}_s{seed}{SUF}"
        if os.path.exists(f"{run}/final.pt"):
            print("done already:", run); continue
        !python scripts/run_sft.py --init {DELEG} --data {DATA} --seed {seed} --replay {REPLAY}/{m}_s{seed}.jsonl --out {run} --max_minutes 300 --resume {quick}

## 4. Phase 7: the test evaluation (~20–30 min in full mode)
Run once. Primary endpoint: accuracy at a 10% call budget. `*` = the 95% CI excludes 0.

In [ ]:
branches = " ".join(f"--branch standard={s}={STD[s]}" for s in SEEDS)
branches += " " + " ".join(f"--branch {m}={s}={WORK}/runs/sft_{m}_s{s}{SUF}/final.pt" for s in SEEDS for m in METHODS)
!python scripts/final_eval.py --data {DATA} --delegation {DELEG} {branches} --out {FINAL} {quick}

## 5. Figure + files to send back

In [ ]:
from IPython.display import Image, display
if os.path.exists(f"{FINAL}/final_results.png"):
    display(Image(f"{FINAL}/final_results.png"))
print(open(f"{REPLAY}/selection_report.json").read()[:3000])
print("Send back: the printed output of sections 1, 2 and 4 (or download results_table.csv, comparisons.csv, selection_report.json).")